In [ ]:
!nvidia-smi  # check GPU usage (e.g. is there already memory blocked)

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# this is important on our server, as the default /tmp directory is very small.
# so if you compile lots of models, it will eventually fill up.
# if you are working on your own machine, you might want/have to remove this!
if os.path.isdir("/var/tmp"):
    os.environ["TMPDIR"] = "/var/tmp"
    os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/var/tmp"

# Restricted Boltzmann Machine (For Binary Data)

To start our expedition into actual generative models, we look a really "old-school" one!
RBMs are somewhat... restricted, so don't expect great results.

Imports will almost always follow the same pattern.
There are a few main functions/modules we will always need, and the respective lightning module, so we can reload it from checkpoint.
Depending on the model, there may be additional imports for various investigations.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn

from lgm.data import ImageDataModule
from lgm.models.rbm import RBMTrainingModule
from lgm.visualization import ImagePlotter

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

## Part 1: Preamble

### Model
RBMs don't have that many degrees of freedom.
For the network itself, we have:
- Number of hidden units

...and that's it!
More units, of course, lead to a model with more capacity.

Most interesting things happen in the data processing, training setup and generation parameters.

### Data
The model we have implemented is a _binary_ RBM.
This means it's a model for binary data, i.e. 0/1 only.
Few "real" datasets fulfill that condition, in fact none of the ones we usually want to look at do.
The closest is MNIST, but it's not *quite* binary, either;
there are gray values inbetween the "0" background and "1" pencil strokes.
Still, we may _binarize_ the dataset by thresholding at 0.5 without loosing too much quality.
For this reason, we include a `Binarize` transform in the config `yaml`.
If you are curious, you can see what happens if you leave it out.

Spoilers: You will probably see little difference.
Our model will no longer be theoretically sound, but should still work in practice.
Because of this, you *could* also try other datasets, and simply not binarize them.
But RBMs are rather weak models, so don't expect good results.

### Training
While there are different ways to train an RBM, we employ the _positive/negative phase_ training we learned about in class.
The main issue here is that each training step requires model samples for the negative phase, and sampling is a slow, iterative process using Markov Chains.
Thus, expect RBM training to be rather slow.

You should, however, have learned about some alternatives:
_Contrastive Divergence (CD)_ as well as _Persistent Contrastive Divergence (PCD)_.
Both methods change the Markov Chains somewhat, with the goal of allowing for shorter chain lengths.
Note that the `RBMTrainingModule` has a `chain_mode` argument, as well as its own `chain_length` with which you can change the Markov chain length for training only, without touching generation (this is controlled by the core `RBM` module, which also has its own `chain_length` argument for generation).
You should experiment with the different methods and chain lengths!

A word on evaluating training progress:
Even though we are minimizing some kind of objective function in training, this is _not_ suitable as a metric for judging model quality.
Your only choice is to visually judge the generated samples.
We will tackle the issue of evaluation later!

#### More Implementation Notes
We use Algorithm 18.1 in the [Deep Learning Book](https://www.deeplearningbook.org/) as a guideline.
This can also be found in the uploaded lecture slides (slide 28).
A bit more succinct, this can be put as:
- Sample a batch of real data $x$ and compute $\nabla \log \tilde{p}(x)$ (positive phase).
  The gradient is with respect to the model parameters.
- Sample a batch of _generated_ data $\tilde{x}$ and compute $\nabla \log \tilde{p}(\tilde{x})$ (negative phase).
  This is all that the two for-loops in the algorithm do.
- From the model parameters, add the positive phase (increase unnormalized probabilities for real data) and subtract the negative phase (decrease unnormalized probabilities for generated data).

Note that the gradient is a linear operation, so we can "pull it out" of the sum:

$$\nabla \log \tilde{p}(x) - \nabla \log \tilde{p}(\tilde{x}) = \nabla (\log \tilde{p}(x) - \log \tilde{p}(\tilde{x}))$$

The algorithm is set up to _maximize_ the likelihood of the data;
since we usually _minimize_ our objective function, we can just put a minus in front and minimize that.
Thus, we essentially have this loss function:

$$-(\log \tilde{p}(x) - \log \tilde{p}(\tilde{x}))$$

What are we doing about the gradient?
Well, nothing!
We let `torch` take care of it as usual.
Thus, with the little derivation above, we have re-written the somewhat unfamiliar training algorithm into a format that works just like any other model we have seen or will see (compute loss and minimize), at least at a high level.

Now you migh still ask, how do we get $\log \tilde{p}(x)$?
Since the RBM is an energy based model, by definition $\tilde{p}(x) = \exp(-E(x))$ for some energy function $E$.
This means $\log \tilde{p}(x) = -E(x)$.
Simple!
The actual energy function is also given in the lecture slides, and has already been implemented for you in a batched format for efficient training.

### Sampling
What you may see is that the alternative methods work well for training, but generated samples look awful.
This is due to issues like _spurious modes_ for CD:
Essentially, since the model doesn't start its samples for the negative phase from random points, it doesn't really learn how to generate "new data" from nothing.

A big question here is, how do we even start our random samples?
That is, how are the Markov chains initialized?
Our default is to simply initialize each pixel as 0/1 with a probability of 0.5.
However, we also support initializing each pixel via the _data marginal_, which just means the average value over the dataset.
This results in pixels which are "off" most of the time in the data (e.g. background) to be initialized with 0 almost certainly.
On the other hand, pixels that are often "on" in the data will have a high probability to be active from the beginning.
With this initialization, the Markov chains start off somewhat closer to the data, and generations tend to be much better.
Try it out!

Finally, the binary RBM generates binary data, in principle.
This often leads to ugly "salt-and-pepper" noise in the generations, as due to the large number of pixels, _some_ low-probability pixels will usually activate, and some high-probability ones may not.
Instead, it's often better to simply plot the _probabilities_ for each pixel.
This leads to clean samples with no random noise.
You can control this via an argument to the `generate` function (see below).

## Part 2: Reloading

In [ ]:
dataset = "mnist"
name = "baseline"
version = 0

check_path = f"lightning_logs/rbm/{dataset}/{name}/version_{version}/checkpoints/last.ckpt"

data_module = ImageDataModule.load_from_checkpoint(check_path)
data_module.prepare_data()
data_module.setup()

plotter = ImagePlotter(transform=data_module.normalizer.invert)

In [ ]:
lightning_module = RBMTrainingModule.load_from_checkpoint(check_path)
model = lightning_module.model.eval()
model.compile()
device = lightning_module.device
print("Device is", device)

Unfortunately I have not found a more elegant way of doing the below.
This replicates setting the "start sampler" for the model using statistics from the data.
Since the actual `nn.Module` (`RBM` class) does not have access to the data, this is normally done by the `RBMTrainingModule` before training starts.
Bu we cannot call the `before_fit` method of `lightning_module` here, because it will crash due there being no `Trainer` object (which usually handles training).

In [ ]:
if lightning_module.marginal_chain_init:
    pixel_averages = data_module.get_statistics(mode="dim")[0]
    flat_probs = pixel_averages.view(-1)
    model.start_sampler = torch.distributions.Bernoulli(probs=flat_probs)

    plt.imshow(pixel_averages.numpy().transpose(1,2,0), cmap="Greys")
    plt.title("Per-pixel average")
    plt.show()
else:
    print("Model does to have data-based start sampler.")

## Part 3: Generations

Here you can experiment with passing different values for `chain_length` to the `generate` function.
How long do chains need to be to produce proper results?
Obviously, shorter chains are faster, but there is probably some lower limit below which results start degrading.
Also see what happens when setting `return_probs=False`!

In [ ]:
n_rows = 10

with torch.inference_mode():
    generated = model.generate(n_rows**2, return_probs=True)

_ = plotter.image_grid(generated,
                       title=f"Generations after {model.chain_length} steps",
                       n_rows=n_rows)

### Analysis

Due to the simplicity of the RBM model, we can attempt some rudimentary introspection.
There is one bias per pixel, so we can shape it into an image and plot.
This shows the general "preference" of the model to activate certain pixels.

In [ ]:
bias_image = model.bias_v.detach().cpu().numpy().reshape((32, 32))
plt.imshow(bias_image, cmap="Greys")
plt.colorbar()
plt.show()

Similarly, each hidden unit is connected to each visible unit, so we can do the same thing.
Each hidden unit encodes an input pattern that would maximally activate it (give it the highest probability to be 1).
Here, red color indicates a positive connection, blue color a negative one.

Unfortunately, this is rarely interpretable.
Most units seem to react either to "blobs" in the image, or have seemingly random patterns all over.

In [ ]:
colormap_norm = "local"

weight_images = model.w_v_to_h.detach().cpu().numpy().T.reshape((-1, 32, 32))

absmax = abs(weight_images).max()
plt.figure(figsize=(12, 12))
# here, only for a selection of units
for ind in range(144):
    if colormap_norm == "local":
        absmax = abs(weight_images[ind]).max()
    plt.subplot(12, 12, ind+1)
    plt.imshow(weight_images[ind], vmin=-absmax, vmax=absmax, cmap="coolwarm")
    plt.axis("off")
plt.show()

### Visualizing Markov Chains

Finally, we can look at the entire Markov chain underlying a generation and create an animation from it! 
The code is a bit awkward, because we have to concatenate all the images into one big image.
Another way would be via subplots and figures, but then we'd have to convert the figures back to numpy arrays for the animation code, which is not fun.

Note: We use a frame rate of 30 for the animation below;
this is quite fast and makes it difficult to see the individual steps.
Feel free to reduce this to something like 1 or 2 to see a bit more detail.
Each frame is one "state" in the Markov chain.
And keep in mind that technically, _each image_ is actually one Markov chain!
We are just running many in parallel.

In [ ]:
chain_length = 500
n_rows = 10

with torch.inference_mode():
    generated, markov_chain = model.generate(n_rows**2, chain_length, return_chain=True)

_ = plotter.image_grid(generated,
                       title=f"Generations after {chain_length} steps",
                       n_rows=n_rows)

In [ ]:
pixels_h, pixels_w = generated.shape[2:]

figure_images = []
for step_ind, step_image_batch in enumerate(markov_chain):
    step_image_batch = step_image_batch.view(-1, pixels_h * n_rows, pixels_w).cpu().numpy()
    step_image_batch = np.concatenate(step_image_batch, axis=1)
    figure_images.append(step_image_batch[None, :, :])  # add channel axis

as_tensor = torch.tensor(np.array(figure_images))
plotter.make_animation(f"animations/rbm_chain_{dataset}_{name}",
                       as_tensor,
                       format="webp",
                       frame_rate=30,
                       invert_colors=True)

Later, we will also see how to evaluate generative models in a principled fashion. That way, we could compare, for example:
- Different training methods like CD or PCD.
- Different chain lengths.
- Using the data-based start sampler vs. uniform.
- Using final visible _probabilities_ as our samples vs. random binary samples.
- ...